In [2]:
import os
import glob
import geopandas as gpd

# 1. REMPLACEZ PAR LE CHEMIN DE VOTRE DOSSIER UDALPLAN
dossier_udalplan = r"C:\Users\pierr\Desktop\Projet1_Gipuzkoa\UDALPLAN_2024_10000_ETRS89"

# On cherche tous les fichiers qui se terminent par .shp dans ce dossier
fichiers_shp = glob.glob(os.path.join(dossier_udalplan, "*.shp"))

print(f"🔍 Scan démarré : {len(fichiers_shp)} fichiers Shapefile trouvés.\n")

# Mots-clés que l'on cherche dans les colonnes pour repérer le bon fichier
mots_cles = ['USO', 'CLAS', 'CALIF', 'MUN', 'UDAL', 'AFUN']

# On boucle sur chaque fichier
for fichier in fichiers_shp:
    nom_fichier = os.path.basename(fichier)
    
    try:
        # On ne charge QUE les 5 premières lignes (rows=5) pour que ça prenne 1 seconde par fichier !
        gdf_test = gpd.read_file(fichier, rows=5)
        colonnes = gdf_test.columns.tolist()
        
        # On vérifie si l'une des colonnes contient un de nos mots-clés
        est_interessant = any(mot in col.upper() for col in colonnes for mot in mots_cles)
        
        if est_interessant:
            print(f"✅ FICHIER PROMETTEUR : {nom_fichier}")
            print(f"   Colonnes trouvées : {colonnes}")
            print("-" * 70)
            
    except Exception as e:
        # Si un fichier est corrompu ou vide, on passe au suivant sans planter
        print(f"⚠️ Impossible de lire {nom_fichier} : {e}")

print("\n🏁 Scan terminé !")

🔍 Scan démarré : 18 fichiers Shapefile trouvés.

✅ FICHIER PROMETTEUR : ct_udal_afun_10000.shp
   Colonnes trouvées : ['AFUN', 'AFUN_EU', 'SUM_SUP_TM', 'SUM_POBLAC', 'SUM_LICRES', 'SUM_R1_SUP', 'SUM_R1_EXI', 'SUM_R1_NUE', 'SUM_R1_VPO', 'SUM_R1_VT', 'SUM_R1_EDI', 'SUM_R1B_SU', 'SUM_R1B_EX', 'SUM_R1B_NU', 'SUM_R1B_VP', 'SUM_R1B_VT', 'SUM_R1B_ED', 'SUM_R2_SUP', 'SUM_R2_EXI', 'SUM_R2_NUE', 'SUM_R2_VPO', 'SUM_R2_VT', 'SUM_R2_EDI', 'SUM_R3_SUP', 'SUM_R3_EXI', 'SUM_R3_NUE', 'SUM_LICAAE', 'SUM_I1_SUP', 'SUM_I1_OCU', 'SUM_I1_LIB', 'SUM_I1_EDI', 'SUM_I1B_SU', 'SUM_I1B_OC', 'SUM_I1B_LI', 'SUM_I1B_ED', 'SUM_I2_SUP', 'SUM_I2_OCU', 'SUM_I2_LIB', 'SUM_I2_EDI', 'SUM_I3_SUP', 'SUM_I3_OCU', 'SUM_I3_LIB', 'POBLACIONH', 'POBLACIONM', 'SUM_EQUIP', 'SUM_LIBRES', 'SUM_INFRAS', 'SUM_VIARIO', 'SUM_PUERTO', 'SUM_AEROPU', 'SUM_FERROC', 'SUM_TAV', 'SUM_CAUCES', 'SUM_MOVILI', 'ESPPRO', 'MEJAMB', 'FOREST', 'AGROG', 'PASTIZ', 'SINVOC', 'AGSUP', 'EXTRAC', 'ALTVALESTR', 'SUM_R1_PRE', 'SUM_R2_PRE', 'SUM_R3_PRE', 'SUM_R

In [4]:
import geopandas as gpd
import os

# 1. PARAMÈTRES 
dossier_udalplan = r"C:\Users\pierr\Desktop\Projet1_Gipuzkoa\UDALPLAN_2024_10000_ETRS89" # Assurez-vous que c'est le bon dossier
dossier_sortie = r"C:\Users\pierr\Desktop\Projet1_Gipuzkoa\GAMA_Files_SanSebastian"

# On crée le dossier de sortie spécifique pour San Sebastian s'il n'existe pas
os.makedirs(dossier_sortie, exist_ok=True)

# 2. FONCTION DE DÉCOUPAGE INTELLIGENTE
def decouper_et_sauvegarder(nom_fichier_source, nom_fichier_sortie):
    chemin_source = os.path.join(dossier_udalplan, nom_fichier_source)
    chemin_sortie = os.path.join(dossier_sortie, nom_fichier_sortie)
    
    if os.path.exists(chemin_source):
        print(f"⏳ Traitement de {nom_fichier_source}...")
        gdf = gpd.read_file(chemin_source)
        
        # ASTUCE : On cherche toute ville contenant le mot "Donostia" 
        # pour éviter les bugs liés à "Donostia / San Sebastián"
        gdf_ville = gdf[gdf['MUNICIPIO'].str.contains('Donostia', case=False, na=False)]
        
        if not gdf_ville.empty:
            # GAMA préfère parfois que l'on supprime les données vides ou bizarres avant d'exporter
            gdf_ville.to_file(chemin_sortie)
            print(f"✅ Succès : {len(gdf_ville)} polygones exportés vers {nom_fichier_sortie}")
        else:
            print(f"⚠️ Aucun polygone trouvé pour Donostia dans ce fichier.")
    else:
        print(f"❌ Fichier source introuvable : {nom_fichier_source}")

# 3. LANCEMENT DU DÉCOUPAGE
print("--- ✂️ EXTRACTION DES ZONES DE DONOSTIA / SAN SEBASTIÁN POUR GAMA ---")

# 1. Maisons (Résidentiel)
decouper_et_sauvegarder("ct_udal_resi_10000.shp", "GAMA_Donostia_Maisons.shp")

# 2. Travail (Industriel/Économique)
decouper_et_sauvegarder("ct_udal_indus_10000.shp", "GAMA_Donostia_Travail.shp")

# 3. Loisirs (Espaces libres)
decouper_et_sauvegarder("ct_udal_libre_10000.shp", "GAMA_Donostia_Loisirs.shp")

print("\n🎉 Terminé ! Les 3 fichiers de Donostia-San Sebastián sont prêts pour GAMA.")

--- ✂️ EXTRACTION DES ZONES DE DONOSTIA / SAN SEBASTIÁN POUR GAMA ---
⏳ Traitement de ct_udal_resi_10000.shp...
✅ Succès : 166 polygones exportés vers GAMA_Donostia_Maisons.shp
⏳ Traitement de ct_udal_indus_10000.shp...
✅ Succès : 28 polygones exportés vers GAMA_Donostia_Travail.shp
⏳ Traitement de ct_udal_libre_10000.shp...


C:\Users\pierr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
C:\Users\pierr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pyogrio\raw.py:200: RuntimeWarning: Geometry of polygon of fid 598 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(


✅ Succès : 23 polygones exportés vers GAMA_Donostia_Loisirs.shp

🎉 Terminé ! Les 3 fichiers de Donostia-San Sebastián sont prêts pour GAMA.
